In [71]:
%load_ext autoreload
%autoreload 2
import torch 
from torch import nn
import torchvision
from torchvision import transforms
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from physics_utils import *
from dataset_dataloader import *
import yaml
from torch.utils.data import WeightedRandomSampler
from torchvision import models
from torch.cuda.amp import GradScaler, autocast
import yaml
from tqdm import tqdm



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [73]:
#temp_train_ds=DatasetMaker('../data/train_data.csv',transforms=transforms.Compose([FftTransform(),transforms.ToTensor()]))
#all=[temp_train_ds[i][0] for i in range(len(temp_train_ds))]
#a=torch.stack(all,dim=0)
#train_mean=a[:,0,:,:].mean().item()
#train_std=a[:,0,:,:].std().item()  ## add these stats to the config file for further use

In [74]:
config_path = '../src/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [75]:
#config['stats']['train_mean'] = round(train_mean, 4)
#config['stats']['train_std'] = round(train_std, 4)
counts_dict = config['train_class_count']
counts = torch.tensor([counts_dict[i] for i in range(len(counts_dict))], dtype=torch.float32)
class_weights = 1.0 / counts
print(f"Calculated Weights: {class_weights}")
config['class_weight']=class_weights.tolist()
config['batch_size']=32
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print("YAML updated")

Calculated Weights: tensor([0.0008, 0.0042, 0.0118, 0.0017])
YAML updated


In [76]:
train_transform = transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                   apply_bilateral=config['fft_params']['apply_bilateral']),
                                      transforms.RandomHorizontalFlip(p=0.5),
                                      transforms.RandomVerticalFlip(p=0.5),
                    # rotations (90 deg increase) to not destroy the fft transform
                                      transforms.RandomChoice([
                                      transforms.RandomRotation((0, 0)),
                                      transforms.RandomRotation((90, 90)),
                                      transforms.RandomRotation((180, 180)),
                                      transforms.RandomRotation((270, 270))]),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

val_transform =transforms.Compose([FftTransform(width=config['fft_params']['notch_width'], notch_depth=config['fft_params']['notch_depth'],
                                                    apply_bilateral=config['fft_params']['apply_bilateral']),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[config['stats']['train_mean']], std=[config['stats']['train_std']])])

In [77]:
train_ds=DatasetMaker(data_csv_path=config['data_path']['train'],transforms=train_transform)
val_ds=DatasetMaker(data_csv_path=config['data_path']['val'],transforms=val_transform)
test_ds=DatasetMaker(data_csv_path=config['data_path']['test'],transforms=val_transform)

**we weight each image label so that we fix the imbalanced labels issue in the dataset where we have majority call of label 0**

In [78]:
# Create a weight for EACH sample based on its class label
sample_weights = []
for idx in range(len(train_ds)):
    label = train_ds.data['label'][idx]
    sample_weights.append(config['class_weight'][label])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_ds),
    replacement=True
)

In [79]:
train_dl=DataLoader(train_ds,batch_size=config['batch_size'],sampler=sampler)
test_dl=DataLoader(test_ds,batch_size=config['batch_size'])
val_dl=DataLoader(val_ds,batch_size=config['batch_size'])

In [80]:
resnet_weights=models.ResNet50_Weights.DEFAULT

In [81]:
class PhotonicResNet50(nn.Module):
    def __init__(self,input_channels=1,class_number=4):
        super().__init__()
        self.input_channel=input_channels
        self.class_num=class_number
        self.weights=models.ResNet50_Weights.DEFAULT
        self.model=models.resnet50(weights=self.weights)
        a=self.model.conv1
        self.model.conv1=nn.Conv2d(in_channels=self.input_channel,out_channels=a.out_channels,kernel_size=a.kernel_size,stride=a.stride,padding=a.padding,bias=False)
        b=self.model.fc
        self.model.fc=nn.Linear(in_features=b.in_features,out_features=self.class_num,bias=True)

    
    def forward(self,x):
        x=self.model(x)
        return x
        
        

In [82]:
model=PhotonicResNet50().to(device=device)

In [83]:
num_of_parameters=sum(p.numel() for p in model.parameters())
print('the model parameters count is :',num_of_parameters,'\n ============================================ ')

the model parameters count is : 23509956 


In [84]:
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-5,weight_decay=1e-2)
loss_fn=nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',  factor=0.1, patience=10, threshold=1e-4 )

**lets check the output dimensions before we start training**

In [85]:
test_input = torch.randn(1, 1, 224, 224).to(device)
with torch.no_grad():
    output = model(test_input)

print(f"Input Shape: {test_input.shape}")
print(f"Output Shape: {output.shape}") 

Input Shape: torch.Size([1, 1, 224, 224])
Output Shape: torch.Size([1, 4])


**we will utilize mixed precesion for faster performance since i am running it on my local gpu**

In [86]:
def evaluate(model, loader, loss_fn, device):
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = loss_fn(outputs, y)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return val_loss / len(loader), correct / total

In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, device, history):
    scaler = GradScaler() 
    best_val_acc = 0.0
    
    # Define Scheduler: 
    # Reduce LR by factor of 0.1 if Val Loss doesn't improve for 3 epochs.
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        from tqdm import tqdm
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for X, y in loop:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(X)
                loss = loss_fn(outputs, y)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == y).sum().item()
            total_train += y.size(0)
            
            loop.set_postfix(loss=loss.item(), acc=correct_train/total_train)

        # Evaluation
        epoch_loss = running_loss / len(train_loader)
        epoch_train_acc = correct_train / total_train
        val_loss, epoch_val_acc = evaluate(model, val_loader, loss_fn, device)
        

        # monitor val_loss to decide when to drop the LR
        scheduler.step(val_loss)
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(epoch_val_acc)
        history['lr'].append(optimizer.param_groups[0]['lr']) # Track LR changes
        
        print(f"\nSummary -> Loss: {epoch_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']}")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), "best_solar_resnet50.pth")
            print("New Best Model Saved!")

        if (epoch + 1) % 5 == 0:
            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history,
                'best_acc': best_val_acc
            }
            torch.save(checkpoint, f"checkpoint_epoch_{epoch+1}.pth")
        
    return history, model

In [88]:
def get_global_stats(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            probs=F.softmax(outputs, dim=1)
            #convert to numpy for sklearn metrics to use
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probs.cpu().numpy())
    return all_labels, all_preds, all_probabilities
